In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from matplotlib import pyplot as plt
import copy, os, time
from datetime import date
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [3]:
names = locals()

functions

In [4]:
def make_constant_values(df, n_sim):
    """
    输入:
        df: 一维年份序列，例如 shape = (92,)
        n_sim: 模拟次数，例如 10000
    输出:
        shape = (n_sim, len(df))
    """
    values = np.array(df).astype(np.float32)  # shape (years,)
    return np.tile(values, (n_sim, 1))

In [5]:
def make_uncertain_values(df, std, span, run, seed):
    values = np.array([df]) if type(df) == float else np.array(df)
    if len(values) == 92:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))
    elif len(values) == 1:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))

    return gg.astype(np.float32) 

In [6]:
def cal_dem_rate(lft_2D, ):
    run, span = lft_2D.shape
    age_1D = [i for i in range(span)]
    age_2D = np.array([age_1D for i in range(run)])
    lft_3D = []
    age_3D = []
    dem_rate_3D = []
    sur_rate_3D = []
    for i in range(span):
        values_1 = np.full(lft_2D.shape, np.nan)
        values_2 = np.full(lft_2D.shape, np.nan)
        values_3 = np.full(lft_2D.shape, np.nan)
        values_1[:, :(span - i)] = lft_2D[:, :(span - i)]
        values_2[:, i:] = age_2D[:, :(span - i)]
        values_3[:, ]
        lft_3D.append(values_1)
        age_3D.append(values_2)

    lft_3D = np.flip(np.array(lft_3D).transpose(1, 2, 0), axis=2)
    age_3D = np.array(age_3D).transpose(1, 0, 2)
    ccc = norm.cdf(age_3D, loc=lft_3D, scale=lft_3D * 0.2)
    sur_rate_3D = 1 - ccc
    dem_rate_3D = np.full(sur_rate_3D.shape, np.nan)
    
    for i in range(span):
        dem_rate_3D[:, i, i] = 0
        ddd = norm.cdf(age_3D[:, i, i:], loc=lft_3D[:, i, i:], scale=lft_3D[:, i, i:] * 0.2)
        dem_rate_3D[:, i, i+1:] = ddd[:, 1:] - ddd[:, :-1]

    # return age_3D, lft_3D, sur_rate_3D, dem_rate_3D
    return sur_rate_3D.astype(np.float32), dem_rate_3D.astype(np.float32)

In [7]:
def make_stock_array(array_2D):
    # print(array_2D.shape)
    run, span = array_2D.shape
    results_1 = []
    for i in range(span):
        values = np.full(array_2D.shape, np.nan)
        values[:, :(span - i)] = array_2D[:, :(span - i)]
        results_1.append(values)
    results_1 = np.array(results_1)
    results = np.flip(results_1.transpose(1, 2, 0), axis=2)

    return results.astype(np.float32)

In [ ]:
def output_results(var_list, start, end, country):
    span = end - start + 1
    results = []
    # q_values = [0, 2.5, 10, 25, 50, 75, 90, 97.5, 100]
    q_values = [2.5, 34, 50, 66, 97.5]
    for var in var_list:
        results.append(np.percentile(names[var], q=q_values, axis=0))
    
    results = np.array(results).transpose(1, 2, 0)

    a_path = RESULTS / 'material-flows' / f'historical' / f'{country}.xlsx'
    with pd.ExcelWriter(a_path) as writer:
        for i, q in enumerate(q_values):
            gg = pd.DataFrame([], index=np.linspace(start, end, span), columns=var_list)
            gg.loc[:, :] = np.round(results[i], 3)
            gg.to_excel(writer, sheet_name=str(q))

    # return results

In [ ]:
def output_uncertainty_results(var_list, country, seed):
    output_path = RESULTS / 'material-flows' / 'historical-uncertainty' / f'{country}-{seed}.npz'
    save_dict = {}
    for var in var_list:
        save_dict[var] = globals()[var]

    np.savez_compressed(output_path, **save_dict)

prepare input data

In [ ]:
region_country = pd.read_excel(DATA / 'contents.xlsx', sheet_name='region_division')
region_country.loc[28, 'region'] = 'NA'
region_country.loc[171, 'region'] = 'NA'

In [11]:
region_pair = dict({1: 'NA', 2:'LAC', 3:'EU', 4:'CIS', 5:'AF', 6:'ME', 7:'IN', 8:'CJK', 9:'DAO', 10:'DA'})

In [ ]:
df_cement_production = pd.read_csv(DATA / 'production-142.csv', index_col=0)
economies_142 = df_cement_production.columns.tolist()

In [ ]:
df_cement_export = pd.read_csv(RESULTS / 'trade-data' / 'cement-export-142.csv', index_col=0)
df_cement_import = pd.read_csv(RESULTS / 'trade-data' / 'cement-import-142.csv', index_col=0)
df_clinker_export = pd.read_csv(RESULTS / 'trade-data' / 'clinker-export-142.csv', index_col=0)
df_clinker_import = pd.read_csv(RESULTS / 'trade-data' / 'clinker-import-142.csv', index_col=0)
df_cement_products_export = pd.read_csv(RESULTS / 'trade-data' / 'cement-product-export-142.csv', index_col=0)
df_cement_products_import = pd.read_csv(RESULTS / 'trade-data' / 'cement-product-import-142.csv', index_col=0)
df_limestone_export = pd.read_csv(RESULTS / 'trade-data' / 'limestone-export-142.csv', index_col=0)
df_limestone_import = pd.read_csv(RESULTS / 'trade-data' / 'limestone-import-142.csv', index_col=0)
df_gravel_export = pd.read_csv(RESULTS / 'trade-data' / 'gravel-crushed-stone-export-142.csv', index_col=0)
df_gravel_import = pd.read_csv(RESULTS / 'trade-data' / 'gravel-crushed-stone-import-142.csv', index_col=0)
df_sand_export = pd.read_csv(RESULTS / 'trade-data' / 'other-sand-export-142.csv', index_col=0)
df_sand_import = pd.read_csv(RESULTS / 'trade-data' / 'other-sand-import-142.csv', index_col=0)

In [ ]:
df_concrete_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='concrete_ratio', index_col=0)
df_C15_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C15_ratio', index_col=0)
df_C16_23_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C16_23_ratio', index_col=0)
df_C24_35_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C24_35_ratio', index_col=0)
df_C35_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C35_ratio', index_col=0)
df_cement_content_Pro = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='cement_in_Pro', index_col=0)

In [ ]:
# load GNR params 
GNR_CCR = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='CCR', index_col=0)
GNR_ratio_gypsum = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_gypsum', index_col=0)
GNR_ratio_limestone = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_limestone', index_col=0)
GNR_ratio_pozzolana = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_pozzolana', index_col=0)
GNR_ratio_slag = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_slag', index_col=0)
GNR_ratio_flyash = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_flyash', index_col=0)
GNR_ratio_others = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_others', index_col=0)
GNR_binder_slag = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='binder_slag', index_col=0)
GNR_binder_FAP = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='binder_FAP', index_col=0)

In [ ]:
# sectoral params
df_ratio_RES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_RES', index_col=0)[economies_142]
df_ratio_NRES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_NRES', index_col=0)[economies_142]
df_ratio_CIV = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_CIV', index_col=0)[economies_142]

df_lft_RES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_RES', index_col=0)[economies_142]
df_lft_NRES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_NRES', index_col=0)[economies_142]
df_lft_CIV = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_CIV', index_col=0)[economies_142]

In [ ]:
df_other_params = pd.read_excel(DATA / 'other-params.xlsx', index_col=0)

In [18]:
df_other_params

,NA,LAC,EU,CIS,AF,ME,IN,CJK,DAO,DA
rec_rate,0.036,0.01,0.0072,0.01,0.01,0.01,0.01,0.0001,0.01,0.01
down_rate,0.510,0.24,0.6042,0.24,0.24,0.24,0.24,0.2300,0.24,0.24


In [19]:
var_names = \
[
    # flow data 
    'limestone_extraction',
    'limestone_stock',
    'excess_limestone',
    'limestone_export',
    'limestone_import',
    'RawMat_Ca', 
    'RawMat_Al', 
    'RawMat_Fe', 
    'RawMat_Si',
    'CKD', 
    'clinker_stock',
    'excess_clinker',
    'clinker_production', 
    'clinker_export',
    'clinker_import',
    'clinker_consumption', 
    'SCM_gypsum',
    'SCM_limestone', 
    'SCM_pozzolana', 
    'SCM_slag', 
    'SCM_flyash', 
    'SCM_others', 
    'cement_production',
    'cement_export',
    'cement_import',
    'cement_consumption', 
    'cement_sub_slag', 
    'cement_sub_FAP', 
    'all_binder', 
    'cementPro_export',
    'cementPro_import',
    'binder_in_exportPro', 
    'binder_in_importPro', 
    'binder_for_concrete', 
    'binder_for_mortar', 
    # 'binder_for_concrete_C15', 
    # 'binder_for_concrete_C16_23', 
    # 'binder_for_concrete_C24_35', 
    # 'binder_for_concrete_C35', 
    # 'coarse_for_concrete_C15', 
    # 'coarse_for_concrete_C16_23', 
    # 'coarse_for_concrete_C24_35', 
    # 'coarse_for_concrete_C35', 
    'coarse_RES',
    'coarse_NRES',
    'coarse_CIV',
    'coarse_total',
    'gravel_import',
    'gravel_export',
    'excess_gravel',
    'gravel_stock',
    'gravel_production',
    'gravel_extraction',
    # 'fine_for_concrete_C15', 
    # 'fine_for_concrete_C16_23', 
    # 'fine_for_concrete_C24_35', 
    # 'fine_for_concrete_C35', 
    # 'fine_mortar', 
    'fine_RES',
    'fine_NRES',
    'fine_CIV',
    'fine_total',
    'sand_import',
    'sand_export',
    'excess_sand',
    'sand_stock',
    'sand_production',
    'sand_extraction',

    
    
    'aggregate_virgin', 
    'aggregate_recycled', 
    # 'water_for_concrete_C15', 
    # 'water_for_concrete_C16_23', 
    # 'water_for_concrete_C24_35', 
    # 'water_for_concrete_C35', 
    # 'water_mortar', 
    'water_total',
    # 'concrete_C15_RES', 
    # 'concrete_C15_NRES', 
    # 'concrete_C15_CIV', 
    # 'concrete_C15', 
    # 'concrete_C16_23_RES',
    # 'concrete_C16_23_NRES',
    # 'concrete_C16_23_CIV',
    # 'concrete_C16_23', 
    # 'concrete_C24_35_RES', 
    # 'concrete_C24_35_NRES', 
    # 'concrete_C24_35_CIV', 
    # 'concrete_C24_35', 
    # 'concrete_C35_RES', 
    # 'concrete_C35_NRES', 
    # 'concrete_C35_CIV', 
    # 'concrete_C35', 
    # 'mortar_RES', 
    # 'mortar_NRES',
    # 'mortar_CIV', 
    'concrete', 
    'mortar', 

    'S_binder_for_concrete_RES_C15_annual', 
    'S_binder_for_concrete_RES_C16_23_annual', 
    'S_binder_for_concrete_RES_C24_35_annual',     
    'S_binder_for_concrete_RES_C35_annual', 
    'S_binder_for_concrete_NRES_C15_annual', 
    'S_binder_for_concrete_NRES_C16_23_annual', 
    'S_binder_for_concrete_NRES_C24_35_annual',     
    'S_binder_for_concrete_NRES_C35_annual', 
    'S_binder_for_concrete_CIV_C15_annual', 
    'S_binder_for_concrete_CIV_C16_23_annual', 
    'S_binder_for_concrete_CIV_C24_35_annual',     
    'S_binder_for_concrete_CIV_C35_annual', 
    'S_binder_for_concrete_RES_annual',
    'S_binder_for_concrete_NRES_annual', 
    'S_binder_for_concrete_CIV_annual', 
    'S_binder_for_mortar_RES_annual', 
    'S_binder_for_mortar_NRES_annual', 
    'S_binder_for_mortar_CIV_annual', 
    'S_binder_RES_annual', 
    'S_binder_NRES_annual', 
    'S_binder_CIV_annual', 
    'S_coarse_RES_annual',
    'S_coarse_NRES_annual',
    'S_coarse_CIV_annual',
    'S_fine_for_concrete_RES_annual',
    'S_fine_for_concrete_NRES_annual',
    'S_fine_for_concrete_CIV_annual',
    'S_fine_for_concrete_annual',
    'S_fine_for_mortar_RES_annual',
    'S_fine_for_mortar_NRES_annual',
    'S_fine_for_mortar_CIV_annual',
    'S_fine_for_mortar_annual',
    'S_fine_RES_annual',
    'S_fine_NRES_annual',
    'S_fine_CIV_annual',
    'S_binder_annual', 
    'S_coarse_annual', 
    'S_fine_annual', 
    'S_water_annual', 
    'S_concrete_annual', 
    'S_mortar_annual', 
    # 'S_aggregate_annual', 
    
    # 'S_RES_annual', 
    # 'S_NRES_annual', 
    # 'S_CIV_annual', 
    'S_Total_annual', 

    'O_binder_annual',
    'O_coarse_annual',
    'O_fine_annual',  
    'O_concrete_annual', 
    'O_mortar_annual', 
    'O_Total_annual', 

    'EoLWaste', 
    'EoLWaste_recycled', 
    'EoLWaste_downcycled', 
    'EoLWaste_disposed', 

    # parameter data 
    'r_Ca', 
    'r_Al', 
    'r_Fe', 
    'r_Si', 
    'CCR', 
    'ratio_gypsum',
    'ratio_limestone',
    'ratio_pozzolana',
    'ratio_slag',
    'ratio_flyash', 
    'ratio_others',
    'binder_slag',
    'binder_FAP',
    'concrete_ratio',
    'mortar_ratio',
    'cement_content', 
    'r_C15', 
    'r_C16_23',
    'r_C24_35',
    'r_C35',
    'coarse_to_binder_C15', 
    'coarse_to_binder_C16_23', 
    'coarse_to_binder_C24_35', 
    'coarse_to_binder_C35',
    'fine_to_binder_C15', 
    'fine_to_binder_C16_23',
    'fine_to_binder_C24_35',
    'fine_to_binder_C35', 
    'fine_to_binder_mortar', 
    'water_to_binder_C15',
    'water_to_binder_C16_23', 
    'water_to_binder_C24_35', 
    'water_to_binder_C35', 
    'water_to_binder_mortar',
    'r_RES', 
    'r_NRES', 
    'r_CIV', 
    'lft_RES', 
    'lft_NRES', 
    'lft_CIV', 
    'r_waste', 
    'r_CKD', 


    # stock cohort, 3rd tier of sub-types, for concrete and mortar 
    'S_binder_for_concrete_RES_C15_cohort', 
    'S_binder_for_concrete_RES_C16_23_cohort', 
    'S_binder_for_concrete_RES_C24_35_cohort',
    'S_binder_for_concrete_RES_C35_cohort', 
    'S_binder_for_concrete_NRES_C15_cohort', 
    'S_binder_for_concrete_NRES_C16_23_cohort', 
    'S_binder_for_concrete_NRES_C24_35_cohort',
    'S_binder_for_concrete_NRES_C35_cohort', 
    'S_binder_for_concrete_CIV_C15_cohort', 
    'S_binder_for_concrete_CIV_C16_23_cohort', 
    'S_binder_for_concrete_CIV_C24_35_cohort',
    'S_binder_for_concrete_CIV_C35_cohort', 
    'S_binder_for_concrete_RES_cohort', 
    'S_binder_for_concrete_NRES_cohort', 
    'S_binder_for_concrete_CIV_cohort', 
    'S_binder_for_mortar_RES_cohort', 
    'S_binder_for_mortar_NRES_cohort', 
    'S_binder_for_mortar_CIV_cohort', 
]

In [20]:
start = 1931
end = 2022
span = end - start + 1
run = 5000
# seed = 1

for seed in tqdm([1, 2], desc=f'run for simulations:'):
    # for country in tqdm(['CHN']): 
    for country in tqdm(economies_142): 
        # print(f'{country} is processing ... ')

        region = region_country[region_country['ISO-3'] == country]['region'].values[0]
        if_developed = region_country[region_country['ISO-3'] == country]['developed'].values[0]
        # std = 0.05 if if_developed == 1 else 0.1
        std = 0.05

        # make uncertain value array for production, export, and import. 
        # cement_production = make_uncertain_values(df_cement_production.loc[start:end, country].values, std, span, run, seed)
        # cement_export = make_uncertain_values(df_cement_export.loc[start:end, country].values, std, span, run, seed)
        # cement_import = make_uncertain_values(df_cement_import.loc[start:end, country].values, std, span, run, seed)
        # clinker_export = make_uncertain_values(df_clinker_export.loc[start:end, country].values, std, span, run, seed)
        # clinker_import = make_uncertain_values(df_clinker_import.loc[start:end, country].values, std, span, run, seed)
        # cementPro_export = make_uncertain_values(df_cement_products_export.loc[start:end, country].values, std, span, run, seed)
        # cementPro_import = make_uncertain_values(df_cement_products_import.loc[start:end, country].values, std, span, run, seed)
        # limestone_export = make_uncertain_values(df_limestone_export.loc[start:end, country].values, std, span, run, seed)
        # limestone_import = make_uncertain_values(df_limestone_import.loc[start:end, country].values, std, span, run, seed)
        # gravel_export = make_uncertain_values(df_gravel_export.loc[start:end, country].values, std, span, run, seed)
        # gravel_import = make_uncertain_values(df_gravel_import.loc[start:end, country].values, std, span, run, seed)
        # sand_export = make_uncertain_values(df_sand_export.loc[start:end, country].values, std, span, run, seed)
        # sand_import = make_uncertain_values(df_sand_import.loc[start:end, country].values, std, span, run, seed)

        cement_production = make_constant_values(df_cement_production.loc[start:end, country].values, run)
        cement_export = make_constant_values(df_cement_export.loc[start:end, country].values, run)
        cement_import = make_constant_values(df_cement_import.loc[start:end, country].values, run)
        clinker_export = make_constant_values(df_clinker_export.loc[start:end, country].values, run)
        clinker_import = make_constant_values(df_clinker_import.loc[start:end, country].values, run)
        cementPro_export = make_constant_values(df_cement_products_export.loc[start:end, country].values, run)
        cementPro_import = make_constant_values(df_cement_products_import.loc[start:end, country].values, run)
        limestone_export = make_constant_values(df_limestone_export.loc[start:end, country].values, run)
        limestone_import = make_constant_values(df_limestone_import.loc[start:end, country].values, run)
        gravel_export = make_constant_values(df_gravel_export.loc[start:end, country].values, run)
        gravel_import = make_constant_values(df_gravel_import.loc[start:end, country].values, run)
        sand_export = make_constant_values(df_sand_export.loc[start:end, country].values, run)
        sand_import = make_constant_values(df_sand_import.loc[start:end, country].values, run)


        # make uncertain value array for CCR, composition of SCMs, and cement substitutes. 
        np.random.seed(seed)
        CCR_mode = np.where(GNR_CCR.loc[start:end, country].values > 0.95, 0.95, GNR_CCR.loc[start:end, country]).astype(np.float32)
        CCR = np.random.triangular(np.ones(span)*0.5, CCR_mode, np.ones(span)*0.95, size=(run, span)).astype(np.float32)
        # CCR = make_uncertain_values(GNR_CCR.loc[start:end, country].values, std, span, run, seed)
        ratio_gypsum = make_uncertain_values(GNR_ratio_gypsum.loc[start:end, country].values, std, span, run, seed)
        ratio_limestone = make_uncertain_values(GNR_ratio_limestone.loc[start:end, country].values, std, span, run, seed)
        ratio_pozzolana = make_uncertain_values(GNR_ratio_pozzolana.loc[start:end, country].values, std, span, run, seed)
        ratio_slag = make_uncertain_values(GNR_ratio_slag.loc[start:end, country].values, std, span, run, seed)
        ratio_flyash = make_uncertain_values(GNR_ratio_flyash.loc[start:end, country].values, std, span, run, seed)
        ratio_others = make_uncertain_values(GNR_ratio_others.loc[start:end, country].values, std, span, run, seed)
        binder_slag = make_uncertain_values(GNR_binder_slag.loc[start:end, country].values, std, span, run, seed)
        binder_FAP = make_uncertain_values(GNR_binder_FAP.loc[start:end, country].values, std, span, run, seed)

        # make uncertain value array for the ratio of concrete, mortar, C15, C16-23, C24-35, C35. 
        concrete_ratio = make_uncertain_values(df_concrete_ratio.loc[start:end, country].values, std, span, run, seed)
        mortar_ratio = 100 - concrete_ratio
        r_C15_ = make_uncertain_values(df_C15_ratio.loc[start:end, country].values, std, span, run, seed)
        r_C16_23_ = make_uncertain_values(df_C16_23_ratio.loc[start:end, country].values, std, span, run, seed)
        r_C24_35_ = make_uncertain_values(df_C24_35_ratio.loc[start:end, country].values, std, span, run, seed)
        r_C35_ = make_uncertain_values(df_C35_ratio.loc[start:end, country].values, std, span, run, seed)
        r_C15 = r_C15_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
        r_C16_23 = r_C16_23_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
        r_C24_35 = r_C24_35_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
        r_C35 = r_C35_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)

        # make ... 
        fine_to_binder_C15 = make_uncertain_values(2.542840011, std, span, run, seed)
        fine_to_binder_C16_23 = make_uncertain_values(2.855202053, std, span, run, seed)
        fine_to_binder_C24_35 = make_uncertain_values(2.083908811, std, span, run, seed)
        fine_to_binder_C35 = make_uncertain_values(1.728617340, std, span, run, seed)
        water_to_binder_C15 = make_uncertain_values(0.659834197, std, span, run, seed)
        water_to_binder_C16_23 = make_uncertain_values(0.619416533, std, span, run, seed)
        water_to_binder_C24_35 = make_uncertain_values(0.496178881, std, span, run, seed)
        water_to_binder_C35 = make_uncertain_values(0.395957047, std, span, run, seed)
        coarse_to_binder_C15 = make_uncertain_values(4.182638806, std, span, run, seed)
        coarse_to_binder_C16_23 = make_uncertain_values(3.271740668, std, span, run, seed)
        coarse_to_binder_C24_35 = make_uncertain_values(2.584728036, std, span, run, seed)
        coarse_to_binder_C35 = make_uncertain_values(2.471397967, std, span, run, seed)

        fine_to_binder_mortar = make_uncertain_values(2.720389651, std, span, run, seed)
        water_to_binder_mortar = make_uncertain_values(0.506558494, std, span, run, seed)

        # ... 
        aa = region_country[region_country['ISO-3'] == country]['cement_content_in_products'].values.astype(np.float32)
        cement_content = make_uncertain_values(aa, std, span, run, seed)

        # value array for sector ratio and lifetime 
        r_RES_ = make_uncertain_values(df_ratio_RES.loc[start:end, country].values, std, span, run, seed)
        r_NRES_ = make_uncertain_values(df_ratio_NRES.loc[start:end, country].values, std, span, run, seed)
        r_CIV_ = make_uncertain_values(df_ratio_CIV.loc[start:end, country].values, std, span, run, seed)
        r_RES = r_RES_ / (r_RES_ + r_NRES_ + r_CIV_)
        r_NRES = r_NRES_ / (r_RES_ + r_NRES_ + r_CIV_)
        r_CIV = r_CIV_ / (r_RES_ + r_NRES_ + r_CIV_)

        lft_RES = make_uncertain_values(df_lft_RES.loc[start:end, country].values, std, span, run, seed)
        lft_NRES = make_uncertain_values(df_lft_NRES.loc[start:end, country].values, std, span, run, seed)
        lft_CIV = make_uncertain_values(df_lft_CIV.loc[start:end, country].values, std, span, run, seed)

        r_waste = np.random.triangular(0.01, 0.015, 0.03, size=(run, span)).astype(np.float32)

        r_Ca = make_uncertain_values(1.45402, std, span, run, seed)   # calcareous ratio 
        r_Al = make_uncertain_values(0.09921, std, span, run, seed)    # aluminous ratio 
        r_Fe = make_uncertain_values(0.01798, std, span, run, seed)   # ferrous ratio 
        r_Si = make_uncertain_values(0.09350, std, span, run, seed) # siliceous ratio 

        r_CKD = make_uncertain_values(0.06, std, span, run, seed)

        #********************    CALCULATION     *************************
        cement_consumption_ = cement_production - cement_export + cement_import
        cement_consumption = np.where(cement_consumption_ >= 0, cement_consumption_, 0)
        cement_sub_slag = cement_consumption * binder_slag
        cement_sub_FAP = cement_consumption * binder_FAP
        all_binder = cement_consumption + cement_sub_slag + cement_sub_FAP  # cement_consumption * (1 + binder_slag + binder_FAP)

        binder_in_exportPro = cementPro_export * cement_content
        binder_in_importPro = cementPro_import * cement_content

        gggg = binder_in_importPro - binder_in_exportPro
        binder_for_concrete = (all_binder * concrete_ratio / 100) + gggg
        binder_for_mortar = all_binder * mortar_ratio / 100

        concrete_waste = (binder_for_concrete * r_waste).astype(np.float32)
        mortar_waste = (binder_for_mortar * r_waste).astype(np.float32)

        for app in ['RES', 'NRES', 'CIV']:
            # 1. create create value array for survival and demolition rates 
            names[f'sur_rate_{app}'], names[f'dem_rate_{app}'] = cal_dem_rate(names[f'lft_{app}'])
            # 2. allocate inflow for different application 
            names[f'binder_for_concrete_{app}'] = binder_for_concrete * (1 - r_waste) * names[f'r_{app}']
            names[f'binder_for_mortar_{app}'] = binder_for_mortar * (1 - r_waste) * names[f'r_{app}']
            # 3. calculate outflow and stock 
            initial_array = make_stock_array(names[f'binder_for_concrete_{app}'])
            names[f'S_binder_for_concrete_{app}'] = initial_array * names[f'sur_rate_{app}']
            names[f'O_binder_for_concrete_{app}'] = initial_array * names[f'dem_rate_{app}']

            for strength in ['C15', 'C16_23', 'C24_35', 'C35']: 
                names[f'S_binder_for_concrete_{app}_{strength}'] = names[f'S_binder_for_concrete_{app}'] * make_stock_array(names[f'r_{strength}'])
                names[f'S_fine_for_concrete_{app}_{strength}'] = names[f'S_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'fine_to_binder_{strength}']) 
                names[f'S_coarse_for_concrete_{app}_{strength}'] = names[f'S_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'coarse_to_binder_{strength}']) 
                names[f'S_water_for_concrete_{app}_{strength}'] = names[f'S_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'water_to_binder_{strength}']) 
                
                names[f'O_binder_for_concrete_{app}_{strength}'] = names[f'O_binder_for_concrete_{app}'] * make_stock_array(names[f'r_{strength}'])
                names[f'O_fine_for_concrete_{app}_{strength}'] = names[f'O_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'fine_to_binder_{strength}']) 
                names[f'O_coarse_for_concrete_{app}_{strength}'] = names[f'O_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'coarse_to_binder_{strength}']) 
                names[f'O_water_for_concrete_{app}_{strength}'] = names[f'O_binder_for_concrete_{app}_{strength}'] * make_stock_array(names[f'water_to_binder_{strength}']) 

                names[f'S_concrete_{app}_{strength}'] = np.sum([names[f'S_{material}_for_concrete_{app}_{strength}'] for material in ['binder', 'fine', 'coarse', 'water']], axis=0)
                names[f'O_concrete_{app}_{strength}'] = np.sum([names[f'O_{material}_for_concrete_{app}_{strength}'] for material in ['binder', 'fine', 'coarse', 'water']], axis=0)
            
            names[f'S_fine_for_concrete_{app}'] = np.sum([names[f'S_fine_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'S_coarse_for_concrete_{app}'] = np.sum([names[f'S_coarse_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'S_water_for_concrete_{app}'] = np.sum([names[f'S_water_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'O_fine_for_concrete_{app}'] = np.sum([names[f'O_fine_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'O_coarse_for_concrete_{app}'] = np.sum([names[f'O_coarse_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'O_water_for_concrete_{app}'] = np.sum([names[f'O_water_for_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)

            names[f'S_concrete_{app}'] = np.sum([names[f'S_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
            names[f'O_concrete_{app}'] = np.sum([names[f'O_concrete_{app}_{strength}'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)


            initial_array2 = make_stock_array(names[f'binder_for_mortar_{app}'])
            names[f'S_binder_for_mortar_{app}'] = initial_array2 * names[f'sur_rate_{app}']
            names[f'O_binder_for_mortar_{app}'] = initial_array2 * names[f'dem_rate_{app}']

            names[f'S_fine_for_mortar_{app}'] = names[f'S_binder_for_mortar_{app}'] * make_stock_array(fine_to_binder_mortar)
            names[f'S_water_for_mortar_{app}'] = names[f'S_binder_for_mortar_{app}'] * make_stock_array(water_to_binder_mortar)

            names[f'O_fine_for_mortar_{app}'] = names[f'O_binder_for_mortar_{app}'] * make_stock_array(fine_to_binder_mortar)
            names[f'O_water_for_mortar_{app}'] = names[f'O_binder_for_mortar_{app}'] * make_stock_array(water_to_binder_mortar)

            names[f'S_mortar_{app}'] = np.sum([names[f'S_{material}_for_mortar_{app}'] for material in ['binder', 'fine', 'water']], axis=0)
            names[f'O_mortar_{app}'] = np.sum([names[f'O_{material}_for_mortar_{app}'] for material in ['binder', 'fine', 'water']], axis=0)

        
        S_binder_for_concrete = np.sum([names[f'S_binder_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_binder_for_concrete = np.sum([names[f'O_binder_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_fine_for_concrete = np.sum([names[f'S_fine_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_fine_for_concrete = np.sum([names[f'O_fine_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_coarse_for_concrete = np.sum([names[f'S_coarse_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_coarse_for_concrete = np.sum([names[f'O_coarse_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_water_for_concrete = np.sum([names[f'S_water_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_water_for_concrete = np.sum([names[f'O_water_for_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_concrete = np.sum([names[f'S_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_concrete = np.sum([names[f'O_concrete_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_binder_for_mortar = np.sum([names[f'S_binder_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_binder_for_mortar = np.sum([names[f'O_binder_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_fine_for_mortar = np.sum([names[f'S_fine_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_fine_for_mortar = np.sum([names[f'O_fine_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_water_for_mortar = np.sum([names[f'S_water_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_water_for_mortar = np.sum([names[f'O_water_for_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        S_mortar = np.sum([names[f'S_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)
        O_mortar = np.sum([names[f'O_mortar_{app}'] for app in ['RES', 'NRES', 'CIV']], axis=0)


        S_binder_RES = S_binder_for_concrete_RES + S_binder_for_mortar_RES 
        S_binder_NRES = S_binder_for_concrete_NRES + S_binder_for_mortar_NRES 
        S_binder_CIV = S_binder_for_concrete_CIV + S_binder_for_mortar_CIV 
        S_fine_RES = S_fine_for_concrete_RES + S_fine_for_mortar_RES
        S_fine_NRES = S_fine_for_concrete_NRES + S_fine_for_mortar_NRES
        S_fine_CIV = S_fine_for_concrete_CIV + S_fine_for_mortar_CIV
        S_binder = S_binder_for_concrete + S_binder_for_mortar
        S_fine = S_fine_for_concrete + S_fine_for_mortar
        S_coarse = S_coarse_for_concrete
        S_water = S_water_for_concrete + S_water_for_mortar
        S_RES = S_concrete_RES + S_mortar_RES
        S_NRES = S_concrete_NRES + S_mortar_NRES
        S_CIV = S_concrete_CIV + S_mortar_CIV
        S_Total = S_concrete + S_mortar

        O_binder = O_binder_for_concrete + O_binder_for_mortar
        O_fine = O_fine_for_concrete + O_fine_for_mortar
        O_coarse = O_coarse_for_concrete
        O_water = O_water_for_concrete + O_water_for_mortar
        O_RES = O_concrete_RES + O_mortar_RES
        O_NRES = O_concrete_NRES + O_mortar_NRES
        O_CIV = O_concrete_CIV + O_mortar_CIV
        O_Total = O_concrete + O_mortar


        for app in ['RES', 'NRES', 'CIV']: 
            for strength in ['C15', 'C16_23', 'C24_35', 'C35']:
                for material in ['binder', 'fine', 'coarse', 'water']:
                    names[f'S_{material}_for_concrete_{app}_{strength}_annual'] = np.nansum(names[f'S_{material}_for_concrete_{app}_{strength}'], axis=1)
                    names[f'O_{material}_for_concrete_{app}_{strength}_annual'] = np.nansum(names[f'O_{material}_for_concrete_{app}_{strength}'], axis=1)
                
                names[f'S_concrete_{app}_{strength}_annual'] = np.nansum(names[f'S_concrete_{app}_{strength}'], axis=1)
                names[f'O_concrete_{app}_{strength}_annual'] = np.nansum(names[f'O_concrete_{app}_{strength}'], axis=1)
            
            names[f'S_binder_for_concrete_{app}_annual'] = np.nansum(names[f'S_binder_for_concrete_{app}'], axis=1)
            names[f'O_binder_for_concrete_{app}_annual'] = np.nansum(names[f'O_binder_for_concrete_{app}'], axis=1)

            names[f'S_concrete_{app}_annual'] = np.nansum(names[f'S_concrete_{app}'], axis=1)
            names[f'O_concrete_{app}_annual'] = np.nansum(names[f'O_concrete_{app}'], axis=1)
        
        names[f'S_concrete_annual'] = np.nansum(S_concrete, axis=1)
        names[f'O_concrete_annual'] = np.nansum(O_concrete, axis=1)

        for app in ['RES', 'NRES', 'CIV']: 
            for material in ['binder', 'fine', 'water']:
                names[f'S_{material}_for_mortar_{app}_annual'] = np.nansum(names[f'S_{material}_for_mortar_{app}'], axis=1)
                names[f'O_{material}_for_mortar_{app}_annual'] = np.nansum(names[f'O_{material}_for_mortar_{app}'], axis=1)
            
            names[f'S_mortar_{app}_annual'] = np.nansum(names[f'S_mortar_{app}'], axis=1)
            names[f'O_mortar_{app}_annual'] = np.nansum(names[f'O_mortar_{app}'], axis=1)
        

        S_binder_RES_annual = np.nansum(S_binder_RES, axis=1)
        S_binder_NRES_annual = np.nansum(S_binder_NRES, axis=1)
        S_binder_CIV_annual = np.nansum(S_binder_CIV, axis=1)
        S_binder_annual = np.nansum(S_binder, axis=1)
        S_fine_for_concrete_RES_annual = np.nansum(S_fine_for_concrete_RES, axis=1)
        S_fine_for_concrete_NRES_annual = np.nansum(S_fine_for_concrete_NRES, axis=1)
        S_fine_for_concrete_CIV_annual = np.nansum(S_fine_for_concrete_CIV, axis=1)
        S_fine_for_mortar_RES_annual = np.nansum(S_fine_for_mortar_RES, axis=1)
        S_fine_for_mortar_NRES_annual = np.nansum(S_fine_for_mortar_NRES, axis=1)
        S_fine_for_mortar_CIV_annual = np.nansum(S_fine_for_mortar_CIV, axis=1)
        S_fine_for_concrete_annual = np.nansum(S_fine_for_concrete, axis=1)
        S_fine_for_mortar_annual = np.nansum(S_fine_for_mortar, axis=1)
        S_fine_RES_annual = S_fine_for_concrete_RES_annual + S_fine_for_mortar_RES_annual
        S_fine_NRES_annual = S_fine_for_concrete_NRES_annual + S_fine_for_mortar_NRES_annual
        S_fine_CIV_annual = S_fine_for_concrete_CIV_annual + S_fine_for_mortar_CIV_annual
        S_fine_annual = np.nansum(S_fine, axis=1)
        S_coarse_RES_annual = np.nansum(S_coarse_for_concrete_RES, axis=1)
        S_coarse_NRES_annual = np.nansum(S_coarse_for_concrete_NRES, axis=1)
        S_coarse_CIV_annual = np.nansum(S_coarse_for_concrete_CIV, axis=1)
        S_coarse_annual = np.nansum(S_coarse, axis=1)
        S_water_annual = np.nansum(S_water, axis=1)
        S_concrete_annual = np.nansum(S_concrete, axis=1)
        S_mortar_annual = np.nansum(S_mortar, axis=1)
        S_Total_annual = np.nansum(S_Total, axis=1)
        O_binder_annual = np.nansum(O_binder, axis=1)
        O_fine_annual = np.nansum(O_fine, axis=1)
        O_coarse_annual = np.nansum(O_coarse, axis=1)
        O_water_annual = np.nansum(O_water, axis=1)
        O_concrete_annual = np.nansum(O_concrete, axis=1)
        O_mortar_annual = np.nansum(O_mortar, axis=1)    
        O_Total_annual = np.nansum(O_Total, axis=1)

        concrete = np.diagonal(S_concrete.T)
        mortar = np.diagonal(S_mortar.T)


        fine_RES = np.diagonal(S_fine_RES.T)
        fine_NRES = np.diagonal(S_fine_NRES.T)
        fine_CIV = np.diagonal(S_fine_CIV.T)
        coarse_RES = np.diagonal(S_coarse_for_concrete_RES.T)
        coarse_NRES = np.diagonal(S_coarse_for_concrete_NRES.T)
        coarse_CIV = np.diagonal(S_coarse_for_concrete_CIV.T)
        fine_total = np.diagonal(S_fine.T)
        coarse_total = np.diagonal(S_coarse.T)
        water_total = np.diagonal(S_water.T)

        sand_production_ = fine_total + sand_export - sand_import
        gravel_production_ = coarse_total + gravel_export - gravel_import
        excess_sand = np.where(sand_production_ < 0, -sand_production_, 0)
        excess_gravel = np.where(gravel_production_ < 0, -gravel_production_, 0)
        sand_stock = excess_sand.cumsum(axis=1)
        gravel_stock = excess_gravel.cumsum(axis=1)

        consumed_sand_stock = np.where((sand_stock > 0) & (sand_production_ > 0), np.where(sand_production_ <= sand_stock, sand_production_, sand_stock), 0)
        consumed_gravel_stock = np.where((gravel_stock > 0) & (gravel_production_ > 0), np.where(gravel_production_ <= gravel_stock, gravel_production_, gravel_stock), 0)

        sand_stock = sand_stock - consumed_sand_stock
        gravel_stock = gravel_stock - consumed_gravel_stock

        sand_production = np.where(sand_production_ >= consumed_sand_stock, sand_production_ - consumed_sand_stock, 0)
        gravel_production = np.where(gravel_production_ >= consumed_gravel_stock, gravel_production_ - consumed_gravel_stock, 0)

        # fine_total = sand_production + sand_import - sand_export

        EoLWaste = O_Total_annual
        # rec_rate = df_other_params.loc['rec_rate', region]
        # down_rate = df_other_params.loc['down_rate', region]
        aa = df_other_params.loc['rec_rate', region]
        bb = df_other_params.loc['down_rate', region]
        rec_rate = make_uncertain_values([aa], std, span, run, seed)
        down_rate = make_uncertain_values([bb], std, span, run, seed)

        EoLWaste_recycled = EoLWaste * rec_rate
        EoLWaste_downcycled = EoLWaste * down_rate 
        EoLWaste_disposed = EoLWaste * (1 - rec_rate - down_rate)

        aggregate_recycled_ = EoLWaste_recycled 
        # aggregate_virgin = fine_total + (coarse_total - aggregate_recycled)

        sand_substitution_rate = 0
        sand_substitution = sand_production * sand_substitution_rate

        aggregate_recycled = np.where(aggregate_recycled_ > gravel_production, gravel_production, aggregate_recycled_)
        gravel_extraction = gravel_production - aggregate_recycled 

        sand_extraction = sand_production - sand_substitution
        aggregate_virgin = gravel_extraction + sand_extraction 

        SCM_gypsum = cement_production * ratio_gypsum
        SCM_limestone = cement_production * ratio_limestone
        SCM_pozzolana = cement_production * ratio_pozzolana
        SCM_slag = cement_production * ratio_slag 
        SCM_flyash = cement_production * ratio_flyash
        SCM_others = cement_production * ratio_others 

        clinker_consumption = cement_production * CCR


        clinker_production_ = clinker_consumption + clinker_export - clinker_import
        # clinker_production = np.where(clinker_production_ > 0, clinker_production_, 0)
        excess_clinker = np.where(clinker_production_ < 0, -clinker_production_, 0)
        gg = excess_clinker.cumsum(axis=1)
        clinker_stock = gg.copy()
        clinker_stock[:, 3:] = gg[:, 3:] - gg[:, :-3]
        clinker_production = np.where(clinker_production_ >= clinker_stock, clinker_production_ - clinker_stock, 0)


        CKD = clinker_production * r_CKD  

        RawMat_Ca = clinker_production * r_Ca 
        RawMat_Al = clinker_production * r_Al 
        RawMat_Fe = clinker_production * r_Fe 
        RawMat_Si = clinker_production * r_Si 

        limestone_ = RawMat_Ca + limestone_export - limestone_import
        excess_limestone = np.where(limestone_ < 0, -limestone_, 0)
        
        limestone_stock = excess_limestone.cumsum(axis=1)
        consumed_limestone_stock = np.where((limestone_stock > 0) & (limestone_ > 0), np.where(limestone_ <= limestone_stock, limestone_, limestone_stock), 0)
        limestone_stock = limestone_stock - consumed_limestone_stock
        limestone_extraction = np.where(limestone_ >= consumed_limestone_stock, limestone_ - consumed_limestone_stock, 0)



        # make the cohorts 
        for app in ['RES', 'NRES', 'CIV']:
            names[f'S_binder_for_concrete_{app}_cohort'] = names[f'S_binder_for_concrete_{app}'][:, :, -1]
            names[f'S_binder_for_mortar_{app}_cohort'] = names[f'S_binder_for_mortar_{app}'][:, :, -1]
            for strength in ['C15', 'C16_23', 'C24_35', 'C35']:
                names[f'S_binder_for_concrete_{app}_{strength}_cohort'] = names[f'S_binder_for_concrete_{app}_{strength}'][:, :, -1]

        # output_results(var_names, start, end, country)
        output_uncertainty_results(var_names, country, seed)

        # print(f'{country} finished and output ... ')
        

run for simulations::   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/142 [00:00<?, ?it/s]

  0%|          | 0/142 [00:00<?, ?it/s]

In [21]:
# output_uncertainty_results(var_names, country, seed)

In [22]:
for material in ['binder', 'fine', 'coarse', 'water']:
    print(f'S_{material}_for_concrete')
    for app in ['RES', 'NRES', 'CIV']:
        print(f'S_{material}_for_concrete_{app}')
        for strength in ['C15', 'C16_23', 'C24_35', 'C35']:
            print(f'S_{material}_for_concrete_{app}_{strength}')

for material in ['binder', 'fine', 'water']:
    for app in ['RES', 'NRES', 'CIV']:
        print(f'S_{material}_for_mortar_{app}')


S_binder_for_concrete
S_binder_for_concrete_RES
S_binder_for_concrete_RES_C15
S_binder_for_concrete_RES_C16_23
S_binder_for_concrete_RES_C24_35
S_binder_for_concrete_RES_C35
S_binder_for_concrete_NRES
S_binder_for_concrete_NRES_C15
S_binder_for_concrete_NRES_C16_23
S_binder_for_concrete_NRES_C24_35
S_binder_for_concrete_NRES_C35
S_binder_for_concrete_CIV
S_binder_for_concrete_CIV_C15
S_binder_for_concrete_CIV_C16_23
S_binder_for_concrete_CIV_C24_35
S_binder_for_concrete_CIV_C35
S_fine_for_concrete
S_fine_for_concrete_RES
S_fine_for_concrete_RES_C15
S_fine_for_concrete_RES_C16_23
S_fine_for_concrete_RES_C24_35
S_fine_for_concrete_RES_C35
S_fine_for_concrete_NRES
S_fine_for_concrete_NRES_C15
S_fine_for_concrete_NRES_C16_23
S_fine_for_concrete_NRES_C24_35
S_fine_for_concrete_NRES_C35
S_fine_for_concrete_CIV
S_fine_for_concrete_CIV_C15
S_fine_for_concrete_CIV_C16_23
S_fine_for_concrete_CIV_C24_35
S_fine_for_concrete_CIV_C35
S_coarse_for_concrete
S_coarse_for_concrete_RES
S_coarse_for_concr